In [1]:
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)
np.random.seed(42)

DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), "data")
SEG_TRAIN_PATH = os.path.join(DATA_PATH, "seg_train/seg_train")
SEG_TEST_PATH = os.path.join(DATA_PATH, "seg_test/seg_test")

IMAGE_SIZE = (150, 150)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Dispositivo detectado: {DEVICE}")

Dispositivo detectado: cuda


### Funciones utiles

In [2]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4, path="best_model.pt"):
        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.best_loss = float("inf")
        self.counter = 0

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), self.path)  # guarda el mejor
        else:
            self.counter += 1

        return self.counter >= self.patience  # True = detener

In [3]:
def save_results(
        model_name: str,
        history_df: pd.DataFrame,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        test_loss: float,
        test_accuracy: float
    ) -> None:

    report_df: pd.DataFrame = pd.DataFrame(classification_report(
        y_true, y_pred, output_dict=True, digits=4)
        ).transpose()
    matrix: pd.DataFrame = pd.DataFrame(confusion_matrix(y_true, y_pred))

    results_dir = os.path.join("results", model_name)
    os.makedirs(results_dir, exist_ok=True)

    history_df.to_csv(os.path.join(results_dir, f"history.csv"), index=False)
    report_df.to_csv(os.path.join(results_dir, f"classification_report.csv"))
    pd.DataFrame(matrix).to_csv(os.path.join(results_dir, f"confusion_matrix.csv"), index=False)
    pd.DataFrame(
        {
            "metric": ["test_loss", "test_accuracy"],
            "value": [test_loss, test_accuracy],
        }
    ).to_csv(os.path.join(results_dir, f"summary.csv"), index=False)

In [20]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    losses = []
    predictions = []
    targets = []

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        predictions.extend(logits.argmax(dim=1).detach().cpu().numpy())
        targets.extend(labels.detach().cpu().numpy())

    return float(np.mean(losses)), accuracy_score(targets, predictions)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    losses = []
    predictions = []
    targets = []

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(inputs)
        loss = criterion(logits, labels)

        losses.append(loss.item())
        predictions.extend(logits.argmax(dim=1).cpu().numpy())
        targets.extend(labels.cpu().numpy())

    return float(np.mean(losses)), accuracy_score(targets, predictions), np.array(targets), np.array(predictions)


def train_epochs(
        model, train_loader,
        test_loader,
        criterion,
        optimizer,
        device,
        epochs=20):
    history = []

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_y_true = None
    best_y_pred = None

    early_stopping = EarlyStopping(min_delta=1e-4, path="best_model.pt")
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, y_true, y_pred = evaluate(model, test_loader, criterion, device)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_y_true = y_true
            best_y_pred = y_pred

        if early_stopping.step(val_loss, model):
            print(f"Early stopping en epoch {epoch} | mejor val_loss={best_val_loss:.4f} | mejor val_acc={best_val_acc:.4f}")
            model.load_state_dict(torch.load("best_model.pt"))
            break

        print(f"Epoch {epoch}/{epochs} | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })

    return pd.DataFrame(history), best_y_true, best_y_pred, best_val_loss, best_val_acc


### Carga de Datos

In [5]:
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(root=SEG_TRAIN_PATH, transform=transform)
test_dataset  = datasets.ImageFolder(root=SEG_TEST_PATH,  transform=transform)

nb_classes = len(train_dataset.classes)
print(f"Clases: {train_dataset.classes}")
print(f"Train: {len(train_dataset)} imágenes | Test: {len(test_dataset)} imágenes")

Clases: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Train: 14034 imágenes | Test: 3000 imágenes


In [6]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

### LeNet5

Arquitectura clásica propuesta por LeCun (1998), originalmente diseñada para dígitos escritos a mano en escala de grises.
En esta versión adaptada se realizaron los siguientes cambios para el dataset de imágenes naturales 150×150 RGB:
- Entrada de **3 canales** (RGB) en lugar de 1.
- Capas `Conv2d` con más filtros (32 y 64) y `kernel_size=3` con `padding=1` en lugar de 5 sin padding.
- `BatchNorm2d` tras cada convolución para estabilizar el entrenamiento.
- `MaxPool2d` en lugar de `AvgPool2d`, que retiene las activaciones más fuertes.
- `AdaptiveAvgPool2d` al final de las features para independizar la arquitectura del tamaño de entrada.
- `Dropout` en features (0.25) y en el clasificador (0.5) para reducir overfitting.

In [7]:
class LeNet5(nn.Module):
    def __init__(self, num_classes: int = nb_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 4 * 4, 84),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(84, num_classes),
        )

    def forward(self, inputs):
        outputs = self.features(inputs)    # [B, 64, 4, 4]
        outputs = torch.flatten(outputs, 1)  # [B, 64*4*4]
        outputs = self.classifier(outputs)
        return outputs


model = LeNet5().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [8]:
history_df, y_true, y_pred, val_loss, val_acc = train_epochs(
    model, train_loader, test_loader, criterion, optimizer, DEVICE, epochs=30
)

Epoch 1/30 | Train Loss: 0.9928, Train Acc: 0.6231 | Val Loss: 0.9013, Val Acc: 0.6637
Epoch 2/30 | Train Loss: 0.7604, Train Acc: 0.7272 | Val Loss: 0.6121, Val Acc: 0.7843
Epoch 3/30 | Train Loss: 0.7107, Train Acc: 0.7460 | Val Loss: 0.7109, Val Acc: 0.7413
Epoch 4/30 | Train Loss: 0.6908, Train Acc: 0.7518 | Val Loss: 0.5624, Val Acc: 0.8043
Epoch 5/30 | Train Loss: 0.6604, Train Acc: 0.7605 | Val Loss: 0.6091, Val Acc: 0.7877
Epoch 6/30 | Train Loss: 0.6399, Train Acc: 0.7686 | Val Loss: 0.6933, Val Acc: 0.7363
Epoch 7/30 | Train Loss: 0.6275, Train Acc: 0.7756 | Val Loss: 1.0223, Val Acc: 0.6813
Epoch 8/30 | Train Loss: 0.6043, Train Acc: 0.7820 | Val Loss: 0.4943, Val Acc: 0.8220
Epoch 9/30 | Train Loss: 0.5937, Train Acc: 0.7847 | Val Loss: 0.5372, Val Acc: 0.8057
Epoch 10/30 | Train Loss: 0.5839, Train Acc: 0.7840 | Val Loss: 1.1200, Val Acc: 0.5997
Epoch 11/30 | Train Loss: 0.5692, Train Acc: 0.7905 | Val Loss: 0.4879, Val Acc: 0.8323
Epoch 12/30 | Train Loss: 0.5664, Train A

In [9]:
save_results("lenet5", history_df, y_true, y_pred, val_loss, val_acc)

### Modelo VGGMini

Variante simplificada de la arquitectura VGG, que apila pares de convoluciones `3×3` antes de cada pooling.
A diferencia de LeNet5, cada bloque refuerza la extracción de características con dos capas conv consecutivas, permitiendo aprender patrones más complejos sin aumentar el tamaño del kernel.
- **3 bloques** conv (32 → 64 → 128 filtros), cada uno con dos `Conv2d + BatchNorm + ReLU` y `MaxPool2d`.
- `Dropout(0.25)` al final de cada bloque para regularización espacial.
- `AdaptiveAvgPool2d((4,4))` para hacer la red agnóstica al tamaño de entrada.
- Clasificador con una capa oculta de 256 neuronas y `Dropout(0.5)` antes de la salida.
- Más parámetros que LeNet5, pero mejor capacidad de representación para imágenes naturales.

In [10]:
class VGGMini(nn.Module):
    def __init__(self, num_classes=nb_classes):
        super().__init__()
        self.features = nn.Sequential(
            # Bloque 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
            # Bloque 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
            # Bloque 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [11]:
history_df, y_true, y_pred, val_loss, val_acc = train_epochs(
    model, train_loader, test_loader, criterion, optimizer, DEVICE, epochs=30
)

Epoch 1/30 | Train Loss: 0.4324, Train Acc: 0.8407 | Val Loss: 0.4316, Val Acc: 0.8570
Epoch 2/30 | Train Loss: 0.4335, Train Acc: 0.8406 | Val Loss: 0.4314, Val Acc: 0.8540
Epoch 3/30 | Train Loss: 0.4272, Train Acc: 0.8427 | Val Loss: 0.4457, Val Acc: 0.8550
Epoch 4/30 | Train Loss: 0.4169, Train Acc: 0.8458 | Val Loss: 0.4904, Val Acc: 0.8483
Epoch 5/30 | Train Loss: 0.4242, Train Acc: 0.8427 | Val Loss: 0.4375, Val Acc: 0.8483
Epoch 6/30 | Train Loss: 0.4209, Train Acc: 0.8444 | Val Loss: 0.4150, Val Acc: 0.8610
Epoch 7/30 | Train Loss: 0.4074, Train Acc: 0.8438 | Val Loss: 0.4676, Val Acc: 0.8373
Epoch 8/30 | Train Loss: 0.4095, Train Acc: 0.8469 | Val Loss: 0.4518, Val Acc: 0.8443
Epoch 9/30 | Train Loss: 0.4054, Train Acc: 0.8485 | Val Loss: 0.4764, Val Acc: 0.8493
Epoch 10/30 | Train Loss: 0.4030, Train Acc: 0.8533 | Val Loss: 0.4243, Val Acc: 0.8557
Early stopping en epoch 11


In [12]:
save_results("VGGmini", history_df, y_true, y_pred, val_loss, val_acc)

### Modelo ResNet18

Implementación desde cero de ResNet18, introducida por He et al. (2015). Su innovación principal son las **conexiones residuales (skip connections)**: la entrada de cada bloque se suma directamente a su salida, lo que permite entrenar redes muy profundas sin el problema del *vanishing gradient*.
- **4 grupos de bloques** (`layer1`–`layer4`) con 2 `BasicBlock` cada uno, duplicando los filtros (64 → 128 → 256 → 512) y reduciendo el tamaño espacial con `stride=2`.
- Cada `BasicBlock` aplica dos `Conv2d 3×3 + BatchNorm + ReLU` con un shortcut que iguala dimensiones si cambian.
- `AdaptiveAvgPool2d((1,1))` al final colapsa cada mapa de features a un solo valor por canal.
- La capa `fc` final proyecta los 512 canales al número de clases.
- Al ser más profunda que LeNet5 y VGGMini, requiere más datos para generalizar bien entrenada desde cero.

In [13]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += self.shortcut(x)
        out = self.relu(out)
        return out

In [14]:
class ResNet18(nn.Module):
    def __init__(self, num_classes=nb_classes):
        super(ResNet18, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(BasicBlock, 64, 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.maxpool(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

model = ResNet18().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

In [15]:
history_df, y_true, y_pred, val_loss, val_acc = train_epochs(
    model, train_loader, test_loader, criterion, optimizer, DEVICE, epochs=30
)

Epoch 1/30 | Train Loss: 0.8687, Train Acc: 0.6631 | Val Loss: 0.9174, Val Acc: 0.6557
Epoch 2/30 | Train Loss: 0.6112, Train Acc: 0.7752 | Val Loss: 0.8975, Val Acc: 0.6930
Epoch 3/30 | Train Loss: 0.5361, Train Acc: 0.8050 | Val Loss: 1.0291, Val Acc: 0.6500
Epoch 4/30 | Train Loss: 0.4897, Train Acc: 0.8180 | Val Loss: 0.5435, Val Acc: 0.8007
Epoch 5/30 | Train Loss: 0.4579, Train Acc: 0.8302 | Val Loss: 0.4519, Val Acc: 0.8357
Epoch 6/30 | Train Loss: 0.4377, Train Acc: 0.8405 | Val Loss: 0.4974, Val Acc: 0.8230
Epoch 7/30 | Train Loss: 0.3937, Train Acc: 0.8592 | Val Loss: 1.1902, Val Acc: 0.6263
Epoch 8/30 | Train Loss: 0.3730, Train Acc: 0.8676 | Val Loss: 0.4376, Val Acc: 0.8410
Epoch 9/30 | Train Loss: 0.3532, Train Acc: 0.8722 | Val Loss: 0.5313, Val Acc: 0.8113
Epoch 10/30 | Train Loss: 0.3336, Train Acc: 0.8804 | Val Loss: 0.4782, Val Acc: 0.8253
Epoch 11/30 | Train Loss: 0.3036, Train Acc: 0.8961 | Val Loss: 0.4518, Val Acc: 0.8393
Epoch 12/30 | Train Loss: 0.2880, Train A

In [16]:
save_results("ResNet18", history_df, y_true, y_pred, val_loss, val_acc)

### ResNet18 pre entrenado

Variante de ResNet18 con **transfer learning**: se carga con pesos preentrenados en ImageNet (1.2M imágenes, 1000 clases) y se adapta al problema reemplazando únicamente la última capa `fc`.
- El backbone ya aprendió a detectar bordes, texturas y formas generales — no parte desde cero.
- Solo se reemplaza `model.fc` con `nn.Linear(512, nb_classes)` para las 6 clases del dataset.
- Se usan **learning rates diferenciales**: `lr=1e-4` para el backbone (actualización suave) y `lr=1e-3` para la `fc` (aprende más rápido al ser nueva).
- Esto evita el *catastrophic forgetting*: actualizar los pesos preentrenados muy fuerte destruye el conocimiento adquirido en ImageNet.
- Con pocos datos (~14k imágenes), este enfoque supera ampliamente a entrenar desde cero.

In [21]:
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, nb_classes)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam([
    {"params": [p for n, p in model.named_parameters() if "fc" not in n], "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
])

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [22]:
history_df, y_true, y_pred, val_loss, val_acc = train_epochs(
    model, train_loader, test_loader, criterion, optimizer, DEVICE, epochs=20
)

Epoch 1/20 | Train Loss: 0.3152, Train Acc: 0.8881 | Val Loss: 0.2065, Val Acc: 0.9207
Epoch 2/20 | Train Loss: 0.1414, Train Acc: 0.9508 | Val Loss: 0.2363, Val Acc: 0.9230
Epoch 3/20 | Train Loss: 0.0790, Train Acc: 0.9716 | Val Loss: 0.3378, Val Acc: 0.9023
Epoch 4/20 | Train Loss: 0.0599, Train Acc: 0.9800 | Val Loss: 0.2808, Val Acc: 0.9210
Epoch 5/20 | Train Loss: 0.0585, Train Acc: 0.9795 | Val Loss: 0.2683, Val Acc: 0.9303
Early stopping en epoch 6 | mejor val_loss=0.2065 | mejor val_acc=0.9207


In [23]:
save_results("ResNet18_Pretrained", history_df, y_true, y_pred, val_loss, val_acc)